# Short-term pattern research: **+5% within 10 sessions**

Find features that separate **hits** from **misses**:

- **Label:** `fwd_ret_10 >= MIN_RETURN_PCT` using **calendar** session `T+10` (same as explosive research).
- **Features:** computed only from data **up to** session `T` (no peeking).

Outputs under `$QUANT_DATA_DIR/results/short_term_5pct_10d/`.

Logic: `scripts/short_term_5pct_10d_research.py` · panel loader: `scripts/screen_100pct_1y.py`.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib.util

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

for pkg in ("pandas", "pyarrow", "numpy", "scikit-learn"):
    if importlib.util.find_spec(pkg.replace("-", "_").split(".")[0]) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd

from quant.config import DATA_DIR


def _load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    assert spec and spec.loader
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


screen = _load_module("screen_100pct_1y", REPO_ROOT / "scripts" / "screen_100pct_1y.py")
st = _load_module("short_term_5pct_10d_research", REPO_ROOT / "scripts" / "short_term_5pct_10d_research.py")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("DATA_DIR:", DATA_DIR)

## Config

In [ ]:
FORWARD_SESSIONS = 10
MIN_RETURN_PCT = 5.0
MIN_PRICE = 5.0
EQ_ONLY = True
DATE_START = "2019-01-01"
DATE_END = None
SAMPLE_FRAC = 1.0
RANDOM_SEED = 42
TRADES_EXPORT_MAX = 5000  # None = all rows in trades CSV

MIN_FORWARD_RETURN = st.min_return_from_pct(MIN_RETURN_PCT)
LABEL_COL = f"y_hit_{FORWARD_SESSIONS}d"
RET_COL = f"fwd_ret_{FORWARD_SESSIONS}"
print("Label:", LABEL_COL, f">= {MIN_RETURN_PCT}% over {FORWARD_SESSIONS} sessions")

In [ ]:
panel = screen.load_panel()
print("panel:", panel.shape, panel["Date"].min(), "→", panel["Date"].max())

dataset = st.build_dataset(
    panel,
    forward_sessions=FORWARD_SESSIONS,
    min_forward_return=MIN_FORWARD_RETURN,
    min_price=MIN_PRICE,
    eq_only=EQ_ONLY,
    date_start=DATE_START,
    date_end=DATE_END,
)

if SAMPLE_FRAC < 1.0:
    rng = np.random.default_rng(RANDOM_SEED)
    dataset = dataset.sample(frac=SAMPLE_FRAC, random_state=rng).reset_index(drop=True)

base_rate = dataset[LABEL_COL].mean()
print(f"Rows: {len(dataset):,} | hit rate: {100 * base_rate:.2f}%")
display(dataset[["Date", "Symbol", "Close", RET_COL, LABEL_COL]].head())

## Univariate feature ranking (hit vs miss)

In [ ]:
rank = st.univariate_feature_rank(dataset, label_col=LABEL_COL)
display(rank.head(25))

tag = f"fwd{FORWARD_SESSIONS}_ret{int(MIN_RETURN_PCT)}pct"
ds_path, rank_path, trades_path = st.export_results(
    dataset,
    rank,
    tag=tag,
    forward_sessions=FORWARD_SESSIONS,
    label_col=LABEL_COL,
    max_trades=TRADES_EXPORT_MAX,
    trades_random_seed=RANDOM_SEED,
)
print("Wrote:", ds_path)
print("Wrote:", rank_path)
print("Wrote:", trades_path, f"(cap {TRADES_EXPORT_MAX}, full dataset {len(dataset):,} rows)")

## Optional: multivariate importance (random forest on subsample)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

RF_MAX_ROWS = 400_000
RF_MIN_FEATURE_COVERAGE = 0.50

X, y, rf_diag = st.prepare_rf_sample(
    dataset,
    label_col=LABEL_COL,
    max_rows=RF_MAX_ROWS,
    random_seed=RANDOM_SEED,
    min_coverage=RF_MIN_FEATURE_COVERAGE,
)
feat_cols = list(X.columns)
print("RF diagnostics:", rf_diag)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=200,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
rf.fit(X, y)
imp = pd.DataFrame({"feature": feat_cols, "importance": rf.feature_importances_}).sort_values(
    "importance", ascending=False
)
display(imp)

imp_path = st.OUTPUT_DIR / f"rf_importance_{tag}.csv"
imp.to_csv(imp_path, index=False)
print("Wrote:", imp_path)

## Inspect a top feature by quartile

In [ ]:
TOP_FEATURE = rank.iloc[0]["feature"] if len(rank) else "vol_rvol_20"

s = pd.to_numeric(dataset[TOP_FEATURE], errors="coerce")
q = pd.qcut(s.rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
lift = dataset.assign(bin=q).groupby("bin", observed=True)[LABEL_COL].agg(["mean", "count"])
lift["hit_rate_pct"] = (100 * lift["mean"]).round(2)
display(lift)
print(f"Base rate: {100 * base_rate:.2f}% | feature: {TOP_FEATURE}")